# 05. Secure AI Development

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply secure development practices to AI
- Identify and mitigate security threats
- Harden ML pipelines and deployments

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

# 05. Secure AI Development

## 🚨 THE PROBLEM: We Need Secure AI Systems

**Remember the limitation from the previous notebook?**

We learned GDPR compliance requirements and practices. But we discovered:

**How do we build secure AI systems that protect against attacks and vulnerabilities?**

**The Problem**: Secure AI systems also need:
- ❌ **Security measures** against attacks (adversarial, data poisoning)
- ❌ **Vulnerability management** (identify and fix security issues)
- ❌ **Secure coding practices** (prevent security bugs)
- ❌ **Security testing** (penetration testing, security audits)

**We've learned:**
- ✅ How to use basic data protection (Notebook 1)
- ✅ How to use advanced privacy technologies (Notebook 2)
- ✅ How to use differential privacy (Notebook 3)
- ✅ How to ensure GDPR compliance (Notebook 4)
- ✅ Privacy and compliance practices

**But we haven't learned:**
- ❌ How to **protect against adversarial attacks**
- ❌ How to **manage security vulnerabilities**
- ❌ How to **implement secure coding practices**
- ❌ How to **test for security issues**

**We need secure development practices** to:
1. Protect against adversarial attacks
2. Manage security vulnerabilities
3. Implement secure coding practices
4. Test for security issues

**This notebook solves that problem** by teaching you secure AI development practices!

---

## 📚 Prerequisites (What You Need First)

**BEFORE starting this notebook**, you should have completed:
- ✅ **Example 1: Data Protection** - Understanding basic protection
- ✅ **Example 2: Privacy Technologies** - Understanding PETs
- ✅ **Example 3: Differential Privacy** - Understanding privacy guarantees
- ✅ **Example 4: GDPR Compliance** - Understanding regulatory compliance
- ✅ **Basic Python knowledge**: Functions, data manipulation

**If you haven't completed these**, you might struggle with:
- Understanding why security matters for AI
- Knowing common security vulnerabilities
- Understanding secure coding practices

---

## 🔗 Where This Notebook Fits

**This is the FIFTH core example in Unit 3** - it teaches you secure development! (Two hands-on practice notebooks, 06 and 07, follow before you move on to Unit 4.)

**Why this example LAST?**
- **Before** you can secure systems, you need privacy techniques (Examples 1-3)
- **Before** you can secure systems, you need compliance (Example 4)
- **Before** you can deploy systems, you need security

**Builds on**: 
- 📓 Example 1: Data Protection (basic protection strategies)
- 📓 Example 2: Privacy Technologies (advanced PETs)
- 📓 Example 3: Differential Privacy (privacy guarantees)
- 📓 Example 4: GDPR Compliance (regulatory compliance)

**Leads to**: 
- 📓 Unit 4: Transparency and Accountability (next unit in the course!)

**Why this order?**
1. Secure development provides **security practices** (needed for safe deployment)
2. Secure development teaches **vulnerability management** (critical for protection)
3. Secure development shows **complete security workflow** (development to deployment)

---

## The Story: Building Secure Systems

Imagine you're building a house. **Before** you finish, you need security - locks, alarms, fire safety. **After** implementing security, you have a safe, protected house!

Same with AI: **Before** we have privacy and compliance but may not be secure, now we learn secure development - protect against attacks, manage vulnerabilities, implement secure coding! **After** secure development, we have secure, private, and compliant AI systems!

---

## Why Secure Development Matters

Secure development is essential for ethical AI:
- **Protection**: Protect against adversarial attacks and vulnerabilities
- **Trust**: Build user confidence in secure systems
- **Compliance**: Meet security requirements
- **Risk Mitigation**: Prevent security breaches and data exposure
- **Best Practices**: Follow industry security standards

## Learning Objectives
1. Understand security vulnerabilities in AI systems
2. Learn secure coding practices
3. Understand adversarial attacks and defenses
4. Implement security testing
5. Create security incident response plans
6. Understand secure deployment practices

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

## Part 1: Three Security Threats to AI Systems - Demonstrated

We demonstrate three classic AI-security concerns on a small model:
1. **Adversarial perturbations**: tiny input changes that degrade accuracy
2. **Data poisoning**: corrupted training labels that damage the model
3. **Integrity checking**: detecting that training data was tampered with

In [1]:
# Why: models that ace clean test sets can crumble under small input changes -
# measuring that fragility BEFORE deployment is basic security hygiene.

# Step 1: Adversarial-perturbation robustness test

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("="*80)
print("⚔️  ADVERSARIAL PERTURBATION TEST")
print("="*80)

# Build a synthetic classification task so the security effects are easy to isolate.
np.random.seed(42)
X = np.random.normal(0, 1, (2000, 4))
y = (X @ np.array([0.9, -0.6, 0.4, 0.2]) + np.random.normal(0, 0.3, 2000) > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train the baseline model and record its clean-data accuracy as the reference point.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
clean_acc = accuracy_score(y_test, model.predict(X_test))
print(f"\nAccuracy on clean test data: {clean_acc:.3f}")

print("\nAccuracy under random input perturbations of growing size:")
np.random.seed(0)
# Grow the perturbation size and watch accuracy decay - the gap from clean
# accuracy is the model's fragility at that noise level.
for eps in [0.1, 0.25, 0.5, 1.0]:
    X_adv = X_test + np.random.normal(0, eps, X_test.shape)
    acc = accuracy_score(y_test, model.predict(X_adv))
    print(f"  perturbation std {eps:<4}: accuracy {acc:.3f} "
          f"(drop {clean_acc - acc:+.3f})")
print("\nNote: real adversarial attacks (FGSM, PGD) pick WORST-case directions and")
print("do far more damage at the same size - this random test is a lower bound.")

⚔️  ADVERSARIAL PERTURBATION TEST

Accuracy on clean test data: 0.903

Accuracy under random input perturbations of growing size:
  perturbation std 0.1 : accuracy 0.893 (drop +0.010)
  perturbation std 0.25: accuracy 0.853 (drop +0.050)
  perturbation std 0.5 : accuracy 0.815 (drop +0.088)
  perturbation std 1.0 : accuracy 0.708 (drop +0.195)

Note: real adversarial attacks (FGSM, PGD) pick WORST-case directions and
do far more damage at the same size - this random test is a lower bound.


In [2]:
# Why: attackers can corrupt a model through its TRAINING data - and a simple
# cryptographic fingerprint makes any tampering detectable.

# Step 2: Data poisoning + integrity checking

import hashlib

print("="*80)
print("☠️  DATA POISONING TEST")
print("="*80)
print("\nWe flip a fraction of TRAINING labels and retrain:")
np.random.seed(1)
poison_acc = {}
# Flip an increasing fraction of training labels, retrain, and measure the damage.
for frac in [0.0, 0.05, 0.10, 0.25]:
    y_poisoned = y_train.copy()
    n_flip = int(frac * len(y_poisoned))
    idx = np.random.choice(len(y_poisoned), n_flip, replace=False)
    y_poisoned[idx] = 1 - y_poisoned[idx]
    m = RandomForestClassifier(n_estimators=100, random_state=42)
    m.fit(X_train, y_poisoned)
    poison_acc[frac] = accuracy_score(y_test, m.predict(X_test))
    print(f"  {frac:>4.0%} labels flipped: clean-test accuracy {poison_acc[frac]:.3f}")
drop25 = poison_acc[0.0] - poison_acc[0.25]
print(f"\nIn this run, RANDOM flipping showed little effect at 5-10% but cost")
print(f"{drop25:.3f} accuracy at 25% - random poisoning damage grows with scale.")
print("TARGETED poisoning (backdoors) is the scarier case: it can succeed with")
print("tiny fractions while overall accuracy still LOOKS fine.")

print("\n" + "="*80)
print("🔏 INTEGRITY CHECK: DETECT TAMPERED TRAINING DATA")
print("="*80)
# Hash the exact bytes of X and y: any single changed label produces a
# completely different fingerprint (see the SHA-256 demo in Notebook 06).
def dataset_fingerprint(X, y):
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(X).tobytes())
    h.update(np.ascontiguousarray(y).tobytes())
    return h.hexdigest()[:16]

fp_original = dataset_fingerprint(X_train, y_train)
print(f"\nFingerprint at data-collection time: {fp_original}")

y_tampered = y_train.copy(); y_tampered[0] = 1 - y_tampered[0]  # 1 label changed
fp_now = dataset_fingerprint(X_train, y_tampered)
print(f"Fingerprint before training:         {fp_now}")
print("MATCH -> safe to train" if fp_now == fp_original
      else "MISMATCH -> data was modified since collection: STOP and investigate")

print("""
Secure AI development practices this demonstrates:
  1. Test robustness to input perturbation BEFORE deployment
  2. Track provenance + validate labels to resist poisoning
  3. Hash datasets (and models!) so tampering is detectable
  4. Combine with Unit-3 privacy controls: encryption, DP, access control
""")

☠️  DATA POISONING TEST

We flip a fraction of TRAINING labels and retrain:


    0% labels flipped: clean-test accuracy 0.903


    5% labels flipped: clean-test accuracy 0.907


   10% labels flipped: clean-test accuracy 0.893


   25% labels flipped: clean-test accuracy 0.845

In this run, RANDOM flipping showed little effect at 5-10% but cost
0.058 accuracy at 25% - random poisoning damage grows with scale.
TARGETED poisoning (backdoors) is the scarier case: it can succeed with
tiny fractions while overall accuracy still LOOKS fine.

🔏 INTEGRITY CHECK: DETECT TAMPERED TRAINING DATA

Fingerprint at data-collection time: db1ad88b3d1a08e7
Fingerprint before training:         5010760e985a4d5d
MISMATCH -> data was modified since collection: STOP and investigate

Secure AI development practices this demonstrates:
  1. Test robustness to input perturbation BEFORE deployment
  2. Track provenance + validate labels to resist poisoning
  3. Hash datasets (and models!) so tampering is detectable
  4. Combine with Unit-3 privacy controls: encryption, DP, access control



---

## ➡️ Transition to Unit 4: Transparency and Accountability

### What We've Accomplished

We've completed the five core examples of Unit 3: Privacy and Security! (Notebooks 06 and 07 offer extra hands-on practice with encryption and anonymization - do them before starting Unit 4.) We've learned:
- ✅ How to protect data (encryption, anonymization)
- ✅ How to use advanced privacy technologies (homomorphic encryption, SMPC)
- ✅ How to use differential privacy (mathematical guarantees)
- ✅ How to ensure GDPR compliance (regulatory requirements)
- ✅ How to build secure AI systems (security practices)

### The Next Challenge: Transparency and Accountability

**Privacy and security are important, but they're not the only ethical concerns!**

As we build AI systems, we also need to consider:
- **Transparency**: How do we explain AI decisions?
- **Accountability**: Who is responsible for AI outcomes?
- **Explainability**: How do we make AI understandable?
- **Auditability**: How do we track and verify AI behavior?

**The Problem**: We've learned about privacy and security, but **AI systems also raise transparency and accountability concerns**:
- AI systems make decisions that affect people
- AI systems may be "black boxes" that are hard to understand
- AI systems need to be explainable and auditable
- AI systems need clear accountability mechanisms

**This is exactly what we'll learn in Unit 4: Transparency and Accountability!**

---

## ➡️ Next Steps

**You've completed Unit 3!** Now you understand:
- ✅ How to protect data and ensure privacy
- ✅ How to comply with regulations
- ✅ How to build secure AI systems

**Next Unit**: `unit4-transparency-accountability/`
- Learn about explainable AI (XAI)
- Understand accountability frameworks
- Master transparency requirements
- Build explainable and accountable AI systems

**Congratulations!** 🎉 You've completed Unit 3 and learned how to build private, secure, and compliant AI systems!

## 📚 References

1. Goodfellow, I. J., Shlens, J. & Szegedy, C. (2015). *Explaining and Harnessing Adversarial Examples*. ICLR 2015. <https://arxiv.org/abs/1412.6572>
2. Papernot, N., McDaniel, P., Sinha, A. & Wellman, M. (2016). *Towards the Science of Security and Privacy in Machine Learning*. <https://arxiv.org/abs/1611.03814>
3. NIST (2023). *Artificial Intelligence Risk Management Framework (AI RMF 1.0)*. NIST AI 100-1. <https://www.nist.gov/itl/ai-risk-management-framework>